# Load CSV File into Oracle

## Create Table

In [30]:
import pandas as pd
import os

# ==========================
# 1. Load CSV files
# ==========================
movies_df = pd.read_csv("../data/clean/movies_clean.csv")
boxoffice_df = pd.read_csv("../data/clean/boxoffice_clean.csv")
rt_df = pd.read_csv("../data/clean/rt_reviews_clean.csv")

# ==========================
# 2. CREATE TABLE statements
# ==========================
create_sql = """
-- Drop tables if they exist
BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE rt_reviews PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE box_office PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movies PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

-- Drop tables if they exist
BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movie_genres PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

-- Create movies table
CREATE TABLE movies (
    tconst VARCHAR2(20) PRIMARY KEY,
    primary_title VARCHAR2(4000) NOT NULL,
    norm_title VARCHAR2(4000) NOT NULL,
    title_type VARCHAR2(50),
    start_year NUMBER(4),
    runtime_minutes NUMBER,
    imdb_rating NUMBER(3,1)
);

-- Movie genres table (1-to-many relationship)
CREATE TABLE movie_genres (
    tconst VARCHAR2(20) NOT NULL,
    genre VARCHAR2(100) NOT NULL,
    FOREIGN KEY (tconst) REFERENCES movies(tconst) ON DELETE CASCADE
);

-- Create box_office table
CREATE TABLE box_office (
    norm_title VARCHAR2(4000) NOT NULL,
    genre VARCHAR2(100),
    year NUMBER(4) NOT NULL,
    worldwide_revenue NUMBER,
    PRIMARY KEY (norm_title, year)
);

-- Create rt_reviews table
CREATE TABLE rt_reviews (
    norm_title VARCHAR2(4000) NOT NULL,
    year NUMBER(4) NOT NULL,
    critics_consensus VARCHAR2(4000),
    review_date DATE,
    tomatometer_rating NUMBER,
    PRIMARY KEY (norm_title, year, review_date)
);
"""

with open("../sql/create_tables.sql", "w") as f:
    f.write(create_sql)

## Insert movies table

In [19]:
# Load movies csv file into movies table
import pandas as pd

# Load CSV
df = pd.read_csv("../data/clean/movies_clean.csv")

table_name = "movies"
sql_lines = []


def safe_int(value):
    """Convert numeric string to int or return NULL if missing."""
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return int(value)

def safe_float(value):
    """Convert numeric string to float or return NULL if missing."""
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return float(value)

for _, row in df.iterrows():
    # Handle numbers safely
    start_year = safe_int(row.get("startYear"))
    runtime = safe_int(row.get("runtimeMinutes"))
    imdb_rating = safe_float(row.get("averageRating"))
    tomatometer = safe_float(row.get("tomatometer"))

    # Handle strings and escape quotes
    primary_title = row["primaryTitle"].replace("'", "''") if pd.notna(row["primaryTitle"]) else ""
    norm_title = row["clean_title"].replace("'", "''") if pd.notna(row.get("clean_title")) else ""
    title_type = row["titleType"].replace("'", "''") if pd.notna(row["titleType"]) else ""

    sql_line = f"""INSERT INTO movies (
    tconst, primary_title, norm_title, title_type, start_year, runtime_minutes, imdb_rating
) VALUES (
    '{row["tconst"]}', '{primary_title}', '{norm_title}', '{title_type}', {start_year}, {runtime}, {imdb_rating}
);"""
    sql_lines.append(sql_line)

sql_lines.append("\nCOMMIT;")

# Save SQL file
with open("../sql/movies_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("movies_inserts.sql generated successfully!")

movies_inserts.sql generated successfully!


## Insert movie_genres table

In [20]:
df = pd.read_csv("../data/clean/movies_clean.csv")

sql_lines = []

for _, row in df.iterrows():

    if pd.isna(row["genres"]) or row["genres"] == "\\N":
        continue

    genres = row["genres"].split(",")

    for g in genres:

        genre = g.strip().replace("'", "''")

        sql = f"""
INSERT INTO movie_genres (tconst, genre)
VALUES ('{row["tconst"]}', '{genre}');
"""

        sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movie_genres_inserts.sql", "w") as f:
    f.write("\n".join(sql_lines))

print("movie_genres_inserts.sql generated successfully!")

movie_genres_inserts.sql generated successfully!


## Insert box_office table

In [35]:
import pandas as pd

# Load CSV
df = pd.read_csv("../data/clean/boxoffice_clean.csv")

# Prepare SQL lines
sql_lines = []

# Optional: disable substitution in SQL*Plus
sql_lines.append("SET DEFINE OFF;\n")

# Helper to handle numeric values safely
def safe_number(value):
    if pd.isna(value) or value == "\\N":
        return "NULL"
    # Remove commas if numbers are strings like "1,234,567"
    if isinstance(value, str):
        value = value.replace(",", "")
    try:
        return float(value)
    except:
        return "NULL"

def safe_string(value):
    if pd.isna(value) or value == "":
        return "NULL"
    return "'" + value + "'"

# Generate INSERT statements
for _, row in df.iterrows():
    norm_title = row["clean_title"].replace("'", "''") if pd.notna(row["clean_title"]) else ""
    year = int(row["Year"]) if pd.notna(row["Year"]) else "NULL"
    worldwide = safe_number(row.get("$Worldwide"))
    genre = safe_string(row.get("Genres"))
    
    sql_line = f"""INSERT INTO box_office (
    norm_title, genre, year, worldwide_revenue
) VALUES (
    '{norm_title}', {genre}, {year}, {worldwide}
);"""
    
    sql_lines.append(sql_line)

sql_lines.append("\nCOMMIT;")

# Save to SQL file
with open("../sql/boxoffice_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("boxoffice_inserts.sql generated successfully!")

boxoffice_inserts.sql generated successfully!


## Insert rt_reviews table

In [36]:
import pandas as pd

# Load CSV
df = pd.read_csv("../data/clean/rt_reviews_clean.csv")

sql_lines = ["SET DEFINE OFF;"]

def safe_string(val):
    if pd.isna(val) or val == "":
        return "NULL"
    # Manually escape single quotes by doubling them
    val_str = str(val)
    if "'" in val_str:
        val_str = val_str.replace("'", "''")
    return "'" + val_str + "'"

def safe_number(val):
    if pd.isna(val) or val == "":
        return "NULL"
    return val

def safe_date(val):
    if pd.isna(val) or val == "":
        return "NULL"
    return "TO_DATE('" + str(val) + "', 'YYYY-MM-DD')"

for _, row in df.iterrows():
    norm_title = safe_string(row.get("norm_title"))
    year = safe_number(row.get("year"))
    consensus = safe_string(row.get("critics_consensus"))
    review_date = safe_date(row.get("review_date"))
    tomatometer = safe_number(row.get("tomatometer_rating"))

    sql = "INSERT INTO rt_reviews (norm_title, year, critics_consensus, review_date, tomatometer_rating) VALUES ({}, {}, {}, {}, {});".format(
        norm_title, year, consensus, review_date, tomatometer
    )
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/rt_reviews_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))